# AWR2944 + DCA1000 — Live Acceptance Test

**Cell-by-cell end-to-end acceptance harness for the public `awr2944_dca` API.**

Run each cell **one at a time**, inspect the result, and discuss before continuing.

---

### Ethernet Safety

⚠️ **THIS NOTEBOOK NEVER MODIFIES WINDOWS NETWORK SETTINGS.**

The human user manually configures the dedicated DCA1000 Ethernet adapter.
If the DCA NIC is not ready, this notebook displays `p.eth.instructions()` and **stops**.

### Expected Hardware Configuration

| Item | Value |
|---|---|
| DCA NIC | Ethernet 2 (ASIX AX88179 USB 3.0 to Gigabit Ethernet) |
| Host IPv4 | 192.168.33.30 / 24, no gateway, no DNS |
| DCA1000 IP | 192.168.33.180 |
| Config UDP port | 4096 |
| Data UDP port | 4098 |
| CLI COM port | COM3 (expected) |
| AUX COM port | COM4 (expected) |

---
## Stage 0 — Environment / Package Check

In [ ]:
import sys
import os

print(f"Python:  {sys.version}")
print(f"CWD:     {os.getcwd()}")

import awr2944_dca
print(f"Package: awr2944_dca {awr2944_dca.__version__}")
print(f"Location: {awr2944_dca.__file__}")

from awr2944_dca import RadarProject
print("\n✅ RadarProject imported successfully.")

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\khams008\Documents\awr2944-fmcw-radar\src")

from awr2944_dca import RadarProject

p = RadarProject.open(
    r"C:\Users\khams008\Documents\saeed_radar_acceptance"
)

print(sys.executable)

import awr2944_dca
print(awr2944_dca.__file__)

In [ ]:
import time
from awr2944_dca.headless_serial import AwrUartConnection

def watch_uart(conn, seconds):
    raw = bytearray()
    t0 = time.monotonic()

    while time.monotonic() - t0 < seconds:
        n = conn._serial.in_waiting
        if n:
            raw.extend(conn._serial.read(n))
        time.sleep(0.05)

    return raw.decode("utf-8", errors="replace")


def show_result(name, r):
    print(f"\n{name}")
    print("success   =", r.success)
    print("timed_out =", r.timed_out)
    print("response:")
    for line in r.response_lines:
        print("   ", repr(line))


with AwrUartConnection("COM3", 115200) as conn:

    print("=== PROMPT SYNC ===")
    conn._serial.reset_input_buffer()
    conn._serial.write(b"\n")
    sync = conn.read_until_prompt(timeout=10.0)
    print(repr(sync[-500:]))
    assert "mmwDemo:/>" in sync

    # -----------------------------------------
    # Identify EXACT firmware / processing chain
    # -----------------------------------------
    v = conn.send_command("version", timeout=5.0)
    show_result("=== VERSION ===", v)

    version_text = "\n".join(v.response_lines)

    q = conn.send_command("queryDemoStatus", timeout=5.0)
    show_result("=== DEMO STATUS ===", q)

    # -----------------------------------------
    # CONTROL EXPERIMENT: do absolutely nothing
    # -----------------------------------------
    print("\n=== IDLE CONTROL: NO COMMANDS FOR 8 s ===")
    idle = watch_uart(conn, 8.0)

    if idle:
        print("ASYNC DATA WHILE IDLE:")
        print(repr(idle))
    else:
        print("No asynchronous UART data. Board stayed quiet.")

    if "Starting QSPI Bootloader" in idle:
        print("\n❌ BOARD REBOOTED WHILE IDLE.")
        print("dfeDataOutputMode is NOT the root cause.")
    else:
        print("\n✅ No idle reboot. Now isolating dfeDataOutputMode.")

        f = conn.send_command("flushCfg", timeout=5.0)
        show_result("=== flushCfg ===", f)

        print("\nWatching 2 s after flushCfg...")
        after_flush = watch_uart(conn, 2.0)
        print(repr(after_flush) if after_flush else "No async data.")

        if "Starting QSPI Bootloader" in after_flush:
            print("\n❌ Reboot occurred after flushCfg. STOP.")
        else:
            d = conn.send_command("dfeDataOutputMode 1", timeout=5.0)
            show_result("=== dfeDataOutputMode 1 ===", d)

            print("\n=== WATCHING 8 s AFTER DFE COMMAND ===")
            after_dfe = watch_uart(conn, 8.0)

            if after_dfe:
                print("ASYNC DATA:")
                print(repr(after_dfe))
            else:
                print("No asynchronous UART data.")

            if "Starting QSPI Bootloader" in after_dfe:
                print("\n❌ REBOOT IS STRONGLY TIED TO dfeDataOutputMode.")
            else:
                print("\n✅ NO DFE REBOOT. Earlier attribution was timing-related.")

In [ ]:
import serial, time

print("Opening COM3 and sending NOTHING for 15 seconds...")

ser = serial.Serial("COM3", 115200, timeout=0.1)

raw = bytearray()
t0 = time.monotonic()

while time.monotonic() - t0 < 15:
    n = ser.in_waiting
    if n:
        raw.extend(ser.read(n))
    time.sleep(0.05)

ser.close()

text = raw.decode("utf-8", errors="replace")

print(text)
print()
print("Bootloader occurrences:",
      text.count("Starting QSPI Bootloader"))
print("mmwDemo banner occurrences:",
      text.count("AWR294X MMW Demo"))

In [ ]:
import serial, time

print("Opening COM3 and sending NOTHING for 15 seconds...")

ser = serial.Serial("COM3", 115200, timeout=0.1)

raw = bytearray()
t0 = time.monotonic()

while time.monotonic() - t0 < 15:
    n = ser.in_waiting
    if n:
        raw.extend(ser.read(n))
    time.sleep(0.05)

ser.close()

text = raw.decode("utf-8", errors="replace")

print(text)
print()
print("Bootloader occurrences:",
      text.count("Starting QSPI Bootloader"))
print("mmwDemo banner occurrences:",
      text.count("AWR294X MMW Demo"))

---
## Stage 1 — Fresh Project Creation

Creates (or reopens) a dedicated acceptance experiment directory.

If the directory already exists, you must choose to reuse it or remove it manually.

In [ ]:
from pathlib import Path

EXPERIMENT_DIR = Path(r"C:\Users\khams008\Documents\saeed_radar_acceptance")

if EXPERIMENT_DIR.exists():
    print(f"⚠️  Directory already exists: {EXPERIMENT_DIR}")
    print("    To start fresh, manually rename or delete it, then re-run this cell.")
    print("    Continuing with the existing directory...")
else:
    EXPERIMENT_DIR.mkdir(parents=True)
    print(f"✅ Created experiment directory: {EXPERIMENT_DIR}")

os.chdir(EXPERIMENT_DIR)
print(f"CWD → {os.getcwd()}")

In [ ]:
# Initialize the project (idempotent — safe to re-run)
p = RadarProject.init_here()
print(p)
print(f"\nRoot: {p.root}")
print(f"Name: {p.name}")
print(f"ID:   {p.project_id}")

In [ ]:
# Verify expected scaffolding
expected = ["awr2944.toml", ".awr2944/local.toml", "profiles", "captures"]
print("Project scaffolding check:")
for item in expected:
    exists = (p.root / item).exists()
    icon = "✅" if exists else "❌"
    print(f"  {icon} {item}")

# Also check optional dirs
for item in ["scripts", "notebooks"]:
    exists = (p.root / item).exists()
    icon = "✅" if exists else "—"
    print(f"  {icon} {item} (optional)")

In [ ]:
# Test idempotent reopen
p2 = RadarProject.open_here()
assert p2.root == p.root, f"Root mismatch: {p2.root} != {p.root}"
print(f"✅ open_here() idempotent — same root: {p2.root}")
print(p2)

In [ ]:
import serial, time

print("Opening COM3 and sending NOTHING for 15 seconds...")

ser = serial.Serial("COM3", 115200, timeout=0.1)

raw = bytearray()
t0 = time.monotonic()

while time.monotonic() - t0 < 15:
    n = ser.in_waiting
    if n:
        raw.extend(ser.read(n))
    time.sleep(0.05)

ser.close()

text = raw.decode("utf-8", errors="replace")

print(text)
print()
print("Bootloader occurrences:",
      text.count("Starting QSPI Bootloader"))
print("mmwDemo banner occurrences:",
      text.count("AWR294X MMW Demo"))

---
## Stage 2 — Hardware Discovery

Read-only enumeration of serial ports and network adapters.

In [ ]:
# Serial / COM port discovery
serial_report = p.hardware.discover("serial")
serial_report.print()

In [ ]:
# COM port discovery (may use different underlying scan)
com_report = p.hardware.discover("com")
com_report.print()

In [ ]:
# Network adapter discovery (read-only PowerShell queries)
net_report = p.hardware.discover("network")
net_report.print()

In [ ]:
import serial, time

print("Opening COM3 and sending NOTHING for 15 seconds...")

ser = serial.Serial("COM3", 115200, timeout=0.1)

raw = bytearray()
t0 = time.monotonic()

while time.monotonic() - t0 < 15:
    n = ser.in_waiting
    if n:
        raw.extend(ser.read(n))
    time.sleep(0.05)

ser.close()

text = raw.decode("utf-8", errors="replace")

print(text)
print()
print("Bootloader occurrences:",
      text.count("Starting QSPI Bootloader"))
print("mmwDemo banner occurrences:",
      text.count("AWR294X MMW Demo"))

---
## Stage 3 — Serial Autodetection

Probes only XDS110 candidates. Never probes arbitrary COM devices.

Expected: CLI = COM3, AUX = COM4 (but let autodetection prove it).

In [ ]:
# Dry-run first — inspect without saving
serial_result = p.hardware.autodetect_serial(save=False)
serial_result.print()

In [ ]:
# Save the detected ports to .awr2944/local.toml
serial_result_saved = p.hardware.autodetect_serial(save=True)
serial_result_saved.print()
print(f"\nSaved: {serial_result_saved.saved}")

---
## Stage 4 — TI Toolchain Autodetection

Scans `C:\ti\mmwave_studio_*` for complete toolchain installations.

Expected: `C:\ti\mmwave_studio_03_01_04_04\mmWaveStudio\PostProc`

Expected files: `DCA1000EVM_CLI_Control.exe`, `DCA1000EVM_CLI_Record.exe`, `RF_API.dll`, `cf.json`

In [ ]:
# Dry-run first — inspect without saving
tc_result = p.hardware.autodetect_toolchain(save=False)
tc_result.print()

In [ ]:
# Save the detected toolchain to .awr2944/local.toml
tc_result_saved = p.hardware.autodetect_toolchain(save=True)
tc_result_saved.print()
print(f"\nSaved: {tc_result_saved.saved}")

In [ ]:
# Record the already-existing DCA NIC host IP in this project's machine-local config.
p.config.local.update(
    host_ip="192.168.33.30"
)

print(p.config.local)

In [ ]:
tc_selected = p.hardware.autodetect_toolchain(
    version="03_01_04_04",
    save=False
)
tc_selected.print()

In [ ]:
tc_saved = p.hardware.autodetect_toolchain(
    version="03_01_04_04",
    save=True
)
tc_saved.print()
print(f"Saved: {tc_saved.saved}")

---
## Stage 5 — DCA NIC Readiness

⚠️ **THIS NOTEBOOK DOES NOT CONFIGURE THE NIC.**

Expected clean local state:
- Exactly one adapter owns 192.168.33.30
- Link Up
- /24 prefix
- No default gateway
- DCA 192.168.33.180 in that subnet

If not ready, follow the instructions printed below and configure the dedicated adapter **manually** in Windows.

In [ ]:
eth_status = p.eth.status()
eth_status.print()

In [ ]:
eth_instructions = p.eth.instructions()
eth_instructions.print()

In [ ]:
# Gate: DCA NIC must be ready before proceeding
if not eth_status.ready:
    print("❌ DCA NIC is NOT ready. Configure the dedicated adapter manually.")
    print("   Do NOT proceed until the status above shows ready.")
    raise SystemExit("DCA NIC not ready — stopping.")
else:
    print("✅ DCA NIC configuration ready.")

---
## Stage 6 — DCA Facade

Read-only DCA1000 readiness, configuration, and aliveness checks.

In [ ]:
dca_verify = p.dca.verify()
dca_verify.print()

In [ ]:
dca_status = p.dca.status()
dca_status.print()

In [ ]:
dca_config = p.dca.config()
dca_config.print()

In [ ]:
# Live aliveness check via DcaCli.query_sys_status (not ping)
dca_verify = p.dca.verify()
dca_verify.print()

In [ ]:
# FPGA version
fpga = p.dca.fpga_version()
print(fpga)

---
## Stage 7 — Doctor

Whole-stack preflight authority. **Do NOT proceed to live capture unless doctor indicates readiness.**

In [ ]:
report = p.doctor()
report.print()

In [ ]:
# Gate: doctor must pass before capture
if not report.ready_for_capture:
    print("\n❌ Doctor reports system is NOT ready for capture.")
    print("   Resolve the issues above before proceeding.")
    raise SystemExit("Doctor failed — stopping before capture.")
else:
    print("\n✅ Doctor: system is READY for capture.")

---
## Stage 8 — Hardware-Free Capture Plan

Compile and inspect the capture plan **without touching any hardware**.

Expected invariants:
- Useful/canonical frames = 8
- Guard frames = 1
- Physical frames = 9
- Cube shape = (8, 128, 4, 256)
- hardware_touched = False

In [ ]:
plan = p.capture.plan(
    profile="smoke_v1",
    frames=8,
    guard_frames=1,
)
plan.print()

In [ ]:
dry = p.capture.dry_run(
    profile="smoke_v1",
    frames=8,
    guard_frames=1,
)

# Task 3 invariant: plan.to_dict() == dry_run()
assert plan.to_dict() == dry, "INVARIANT VIOLATION: plan.to_dict() != dry_run()"
print("✅ plan.to_dict() == dry_run()")

In [ ]:
# Verify smoke_v1 invariants
assert plan.canonical_frames == 8, f"Expected 8 canonical frames, got {plan.canonical_frames}"
assert plan.guard_frames == 1, f"Expected 1 guard frame, got {plan.guard_frames}"
assert plan.total_frames == 9, f"Expected 9 total frames, got {plan.total_frames}"
assert plan.cube_shape == (8, 128, 4, 256), f"Expected (8, 128, 4, 256), got {plan.cube_shape}"
assert dry["hardware_touched"] is False, "hardware_touched should be False"

print(f"Canonical frames:  {plan.canonical_frames}")
print(f"Guard frames:      {plan.guard_frames}")
print(f"Total frames:      {plan.total_frames}")
print(f"Cube shape:        {plan.cube_shape}")
print(f"hardware_touched:  {dry['hardware_touched']}")

# Show warnings (expect simultaneous multi-TX warning)
if plan.warnings:
    print(f"\nWarnings ({len(plan.warnings)}):")
    for w in plan.warnings:
        print(f"  ⚠️  {w}")
else:
    print("\nNo warnings.")

print("\n✅ All smoke_v1 plan invariants verified.")

---
## Stage 9 — Explicit Human Live-Capture Gate

### ⚠️ WARNING: The next cell performs REAL HARDWARE I/O.

It will:
1. Open a UART connection to the AWR2944
2. Send SDK CLI configuration commands
3. Initialize the DCA1000 FPGA
4. Trigger radar frame transmission
5. Receive raw ADC data over UDP

**Ensure**:
- AWR2944 is powered on and connected via USB (XDS110)
- DCA1000 is powered on and connected via Ethernet
- Doctor has passed (Stage 7)
- You are ready for RF emission

Set `RUN_LIVE_CAPTURE = True` below and run the next cell.

In [ ]:
# ─── HUMAN GATE ───────────────────────────────────────────────────────
# Change this to True ONLY when you are ready for real hardware capture.
RUN_LIVE_CAPTURE = False
# ─────────────────────────────────────────────────────────────────────

---
## Stage 10 — Real Smoke Capture

Exercises the real public production facade: `p.capture.run_smoke()`.

In [ ]:
if not RUN_LIVE_CAPTURE:
    print("🚫 RUN_LIVE_CAPTURE is False. Set it to True in the cell above to proceed.")
    raise SystemExit("Live capture gate is closed.")

print("🔴 Starting live smoke capture...")
print(f"   Profile: smoke_v1, frames=8, guard_frames=1")
print(f"   Name: acceptance_smoke")
print()

result = p.capture.run_smoke(
    name="acceptance_smoke",
    frames=8,
    guard_frames=1,
)

print(f"\nCapture complete!")
print(f"  Success:    {result.success}")
print(f"  Capture ID: {result.capture.capture_id}")
print(f"  Path:       {result.capture.path}")

---
## Stage 11 — Capture Validation

Inspect the capture result through public APIs.

In [ ]:
# Capture status
cap = result.capture
status = cap.status()
print("Capture status:")
for k, v in status.items():
    print(f"  {k}: {v}")

In [ ]:
# Capture verification
verification = cap.verify()
verification.print()
print(f"\nVerification success: {verification.success}")

In [ ]:
# Production manifest (written by frozen run_capture)
manifest = cap.manifest
if manifest:
    print("Production manifest keys:")
    for k in sorted(manifest.keys()):
        val = manifest[k]
        # Truncate long values for readability
        display = str(val) if len(str(val)) < 80 else str(val)[:77] + "..."
        print(f"  {k}: {display}")
else:
    print("⚠️  No production manifest found (acceptance gap: manifest not written?)")

In [ ]:
# Any warnings from the capture result
if hasattr(result, 'session_result') and result.session_result:
    sr = result.session_result
    print(f"Session result success: {sr.success}")
    if hasattr(sr, 'warnings') and sr.warnings:
        print(f"Warnings:")
        for w in sr.warnings:
            print(f"  ⚠️  {w}")
else:
    print("No session result available.")

---
## Stage 12 — Reopen Capture

Test whether a fresh consumer can retrieve the capture without retaining internal objects.

In [ ]:
# List all captures
all_captures = p.captures.list()
print(f"Total captures in project: {len(all_captures)}")
for c in all_captures:
    print(f"  {c}")

In [ ]:
# Find the newly created capture
reopened = p.captures.get("acceptance_smoke")
print(f"Reopened: {reopened}")
print(f"Path:     {reopened.path}")
assert reopened.capture_id == cap.capture_id, "Capture ID mismatch on reopen"
print("\n✅ Capture successfully reopened via p.captures.get()")

In [ ]:
# Also test latest()
latest = p.captures.latest()
print(f"Latest capture: {latest}")
print(f"  ID:   {latest.capture_id}")
print(f"  Path: {latest.path}")

---
## Stage 13 — Canonical ADC Cube

Use the public `capture.raw` API to load the canonical ADC cube.

Expected: shape = (8, 128, 4, 256), dtype = int16.

In [ ]:
raw = reopened.raw

print(f"Native data path:    {raw.native_path}")
print(f"Canonical data path: {raw.canonical_path}")
print(f"Native bytes:        {raw.native_bytes}")
print(f"Canonical bytes:     {raw.canonical_bytes}")
print(f"Native SHA256:       {raw.native_sha256}")
print(f"Canonical SHA256:    {raw.canonical_sha256}")

In [ ]:
cube = raw.to_cube("canonical")

print(f"Cube shape: {cube.shape}")
print(f"Cube dtype: {cube.dtype}")
print(f"Min value:  {cube.min()}")
print(f"Max value:  {cube.max()}")
print(f"Mean value: {cube.mean():.2f}")

assert cube.shape == (8, 128, 4, 256), f"Expected (8, 128, 4, 256), got {cube.shape}"
assert cube.dtype.name == "int16", f"Expected int16, got {cube.dtype}"
print("\n✅ Canonical cube shape and dtype verified.")

In [ ]:
# Verify SHA256 integrity
disk_hash = raw.compute_sha256("canonical")
print(f"Manifest SHA256: {raw.canonical_sha256}")
print(f"Disk SHA256:     {disk_hash}")

if raw.canonical_sha256:
    assert disk_hash == raw.canonical_sha256, "SHA256 MISMATCH — data corrupted!"
    print("\n✅ SHA256 integrity verified.")
else:
    print("\n⚠️  No SHA256 in manifest (acceptance gap: hash not recorded).")

In [ ]:
# Packet metadata (if available)
pkt = raw.packet_metadata()
if pkt is not None:
    print(f"Packet metadata records: {len(pkt)}")
    if pkt:
        print(f"First record keys: {list(pkt[0].keys())}")
else:
    print("No packet metadata available.")

---
## Stage 14 — Basic DSP

Exercise existing public DSP functionality.

> **Acceptance gap**: DSP access currently requires importing from `awr2944_dca.dsp.*`
> submodules rather than being exposed through a top-level public API on
> `RadarProject` or `RadarCapture`. There is no `p.dsp` or `capture.dsp` accessor.

In [ ]:
# ACCEPTANCE GAP: DSP requires importing internal submodules.
# There is no p.dsp or capture.dsp on the public RadarProject/RadarCapture API.

from awr2944_dca.dsp.config import RadarProfile as DspRadarProfile
from awr2944_dca.dsp.range_fft import compute_range_fft

dsp_profile = DspRadarProfile.from_smoke_v1()
print(f"DSP profile: {dsp_profile}")

range_result = compute_range_fft(cube, dsp_profile)
print(f"\nRange FFT output shape: {range_result.shape}")
print(f"Number of range bins:   {range_result.num_range_bins}")
print(f"Power dB range:         [{range_result.power_db.min():.1f}, {range_result.power_db.max():.1f}] dB")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Average range profile across all frames, chirps, and RX channels
avg_range_profile = range_result.power_db.mean(axis=(0, 1, 2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(avg_range_profile, linewidth=0.8)
ax.set_xlabel("Range Bin")
ax.set_ylabel("Power (dB)")
ax.set_title("Acceptance Smoke — Average Range Profile")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Basic range FFT and plot completed.")

---
## Stage 15 — Acceptance Summary

Fill in PASS / FAIL as you work through each stage.

In [ ]:
checklist = [
    ("Stage 0",  "Environment / package import",      ""),
    ("Stage 1a", "Fresh project init_here()",          ""),
    ("Stage 1b", "open_here() idempotency",            ""),
    ("Stage 2",  "Hardware discovery (serial/net)",    ""),
    ("Stage 3a", "Serial autodetect (save=False)",     ""),
    ("Stage 3b", "Serial autodetect (save=True)",      ""),
    ("Stage 4a", "Toolchain autodetect (save=False)",  ""),
    ("Stage 4b", "Toolchain autodetect (save=True)",   ""),
    ("Stage 5",  "DCA NIC status — ready",             ""),
    ("Stage 6a", "DCA status()",                       ""),
    ("Stage 6b", "DCA config()",                       ""),
    ("Stage 6c", "DCA verify()",                       ""),
    ("Stage 6d", "DCA fpga_version()",                 ""),
    ("Stage 7",  "Doctor — ready for capture",         ""),
    ("Stage 8a", "Capture plan invariants",            ""),
    ("Stage 8b", "plan.to_dict() == dry_run()",        ""),
    ("Stage 10", "Live smoke capture",                 ""),
    ("Stage 11", "Capture verification",               ""),
    ("Stage 12", "Reopen capture via captures.get()",  ""),
    ("Stage 13", "Canonical ADC cube (8,128,4,256)",   ""),
    ("Stage 14", "Basic DSP / range FFT plot",         ""),
]

print(f"{'Stage':<10} {'Check':<40} {'Result':<10}")
print("─" * 60)
for stage, check, result_val in checklist:
    icon = {"PASS": "✅", "FAIL": "❌", "SKIP": "⏭️"}.get(result_val, "⬜")
    print(f"{stage:<10} {check:<40} {icon} {result_val}")

print("\n📋 Fill in the Result column above as you complete each stage.")
print("   Edit the checklist tuples to add 'PASS', 'FAIL', or 'SKIP'.")

---

### Known Acceptance Gaps

| Gap | Description |
|---|---|
| **DSP API** | No public `p.dsp` or `capture.dsp` on `RadarProject`/`RadarCapture`. Must import from `awr2944_dca.dsp.*` submodules directly. |

---

*End of acceptance notebook.*